In [2]:
import sqlite3
import pandas as pd

# Datenbankverbindung erstellen
conn = sqlite3.connect("maschinen.db")
print("Verbindung erfolgreich")

Verbindung erfolgreich


In [3]:
# Tabelle erstellen
conn.execute("""
    CREATE TABLE IF NOT EXISTS maschinen (
        id INTEGER PRIMARY KEY,
        name TEXT,
        temperatur REAL,
        druck REAL,
        vibration REAL,
        status TEXT
    )
""")
conn.commit()
print("Tabelle erstellt")

Tabelle erstellt


In [4]:
# Daten einfügen
conn.execute("INSERT INTO maschinen VALUES (1, 'Pumpe A', 85.0, 5.0, 1.2, 'NORMAL')")
conn.execute("INSERT INTO maschinen VALUES (2, 'Pumpe B', 110.0, 8.5, 3.8, 'KRITISCH')")
conn.execute("INSERT INTO maschinen VALUES (3, 'Pumpe C', 92.0, 6.2, 2.1, 'ERHÖHT')")
conn.execute("INSERT INTO maschinen VALUES (4, 'Pumpe D', 78.0, 4.1, 0.8, 'NORMAL')")
conn.execute("INSERT INTO maschinen VALUES (5, 'Pumpe E', 115.0, 9.0, 4.5, 'KRITISCH')")
conn.commit()
print("Daten eingefügt")

Daten eingefügt


In [5]:
# Alle Maschinen anzeigen
df = pd.read_sql("SELECT * FROM maschinen", conn)
print(df)

   id     name  temperatur  druck  vibration    status
0   1  Pumpe A        85.0    5.0        1.2    NORMAL
1   2  Pumpe B       110.0    8.5        3.8  KRITISCH
2   3  Pumpe C        92.0    6.2        2.1    ERHÖHT
3   4  Pumpe D        78.0    4.1        0.8    NORMAL
4   5  Pumpe E       115.0    9.0        4.5  KRITISCH


In [6]:
# Nur kritische Maschinen
df = pd.read_sql("SELECT * FROM maschinen WHERE status = 'KRITISCH'", conn)
print(df)

   id     name  temperatur  druck  vibration    status
0   2  Pumpe B       110.0    8.5        3.8  KRITISCH
1   5  Pumpe E       115.0    9.0        4.5  KRITISCH


In [7]:
# Nur Maschinen mit Temperatur über 100 Grad
df = pd.read_sql("SELECT name, temperatur, status FROM maschinen WHERE temperatur > 100", conn)
print(df)

      name  temperatur    status
0  Pumpe B       110.0  KRITISCH
1  Pumpe E       115.0  KRITISCH


In [8]:
# Maschinen sortiert nach Temperatur — höchste zuerst
df = pd.read_sql("SELECT name, temperatur, status FROM maschinen ORDER BY temperatur DESC", conn)
print(df)

      name  temperatur    status
0  Pumpe E       115.0  KRITISCH
1  Pumpe B       110.0  KRITISCH
2  Pumpe C        92.0    ERHÖHT
3  Pumpe A        85.0    NORMAL
4  Pumpe D        78.0    NORMAL


In [9]:
# Durchschnitt, Maximum, Minimum der Temperatur
df = pd.read_sql("""
    SELECT
        AVG(temperatur) as durchschnitt,
        MAX(temperatur) as maximum,
        MIN(temperatur) as minimum
    FROM maschinen
""", conn)
print(df)

   durchschnitt  maximum  minimum
0          96.0    115.0     78.0


In [10]:
# Durchschnitt pro Status-Gruppe
df = pd.read_sql("""
    SELECT
        status,
        COUNT(*) as anzahl,
        AVG(temperatur) as avg_temperatur
    FROM maschinen
    GROUP BY status
""", conn)
print(df)

     status  anzahl  avg_temperatur
0    ERHÖHT       1            92.0
1  KRITISCH       2           112.5
2    NORMAL       2            81.5


In [11]:
# Nur nicht-normale Maschinen gruppieren
df = pd.read_sql("""
    SELECT
        status,
        COUNT(*) as anzahl,
        AVG(temperatur) as avg_temperatur,
        MAX(vibration) as max_vibration
    FROM maschinen
    WHERE status != 'NORMAL'
    GROUP BY status
""", conn)
print(df)

     status  anzahl  avg_temperatur  max_vibration
0    ERHÖHT       1            92.0            2.1
1  KRITISCH       2           112.5            4.5
